# Teardown

**WARNING: this permanently deletes every `pamap2-har` AWS resource** (S3 data lake incl. all versions, Feature Group, Glue/Athena, SageMaker Pipeline `pamap2-har-pipeline` (running executions are stopped; pipeline-created models and log streams are removed), SageMaker models/packages/endpoints, Model Card, CloudWatch alarms/dashboard, project log streams). IAM (`LabRole`) and `config/project_config.json` are never touched.

Default is a dry run. To delete: set `DRY_RUN=False` and `CONFIRM` to the bucket name printed below. `KEEP_DATA=True` keeps the bucket and `raw/` + `curated/` objects.

In [3]:
DRY_RUN = False
CONFIRM = "pamap2-har-887757489409-us-east-1"
KEEP_DATA = False

In [2]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3, pandas as pd
from src.pamap2.config import load_config
from src.pamap2.teardown import teardown, discover, plan_order

cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
clients = {n: session.client(n) for n in ("s3", "sagemaker", "glue", "athena", "cloudwatch", "logs")}
print("Bucket (use as CONFIRM):", cfg["bucket"])

Bucket (use as CONFIRM): pamap2-har-887757489409-us-east-1


In [4]:
errs = []
plan = plan_order(discover(clients, cfg, KEEP_DATA, errs))
print(pd.DataFrame([{"kind": r.kind, "name": r.name} for r in plan]).to_string(index=False))
if errs:
    print("\nDISCOVERY FAILURES:")
    print(pd.DataFrame([{"kind": e.kind, "error": e.extra["error"]} for e in errs]).to_string(index=False))

               kind                                                                                     name
           pipeline                                                                      pamap2-har-pipeline
              model                                   pipelines-jlqa6fgd20cq-CreateModel-CreateMo-3hArOibLqL
              model                                   pipelines-h8xqivro7ysc-CreateModel-CreateMo-tlvBF4Xe5S
      model_package              arn:aws:sagemaker:us-east-1:887757489409:model-package/pamap2-har-xgboost/2
      model_package              arn:aws:sagemaker:us-east-1:887757489409:model-package/pamap2-har-xgboost/1
model_package_group                                                                       pamap2-har-xgboost
         model_card                                                                  pamap2-har-xgboost-card
      feature_group                                                                       pamap2-har-windows
         glue_table

In [5]:
rows = teardown(clients, cfg, dry_run=DRY_RUN, confirm=CONFIRM, keep_data=KEEP_DATA)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(summary["status"].value_counts().to_string())
failed = summary[summary["status"] == "failed"]
if len(failed):
    raise RuntimeError(f"{len(failed)} teardown step(s) failed - see summary above")

               kind                                                                                     name  status detail
           pipeline                                                                      pamap2-har-pipeline deleted       
              model                                   pipelines-jlqa6fgd20cq-CreateModel-CreateMo-3hArOibLqL deleted       
              model                                   pipelines-h8xqivro7ysc-CreateModel-CreateMo-tlvBF4Xe5S deleted       
      model_package              arn:aws:sagemaker:us-east-1:887757489409:model-package/pamap2-har-xgboost/2 deleted       
      model_package              arn:aws:sagemaker:us-east-1:887757489409:model-package/pamap2-har-xgboost/1 deleted       
model_package_group                                                                       pamap2-har-xgboost deleted       
         model_card                                                                  pamap2-har-xgboost-card deleted       
      fe